# ETAPA 19 · Matriz masiva Arco 1

Notebook dedicado a **equipo × habilidades × monstruos × adaptación**.

Guardias: no importa DEF legacy como DEF plana; conserva procedencia/seed; no promueve LAB a CANON.


In [ ]:
!rm -rf /content/grulla
!git clone -q -b experiment/combat-stat-contract-v0.1 https://github.com/Shein25/La-Grulla-Blanca.git /content/grulla
%cd /content/grulla/experimentos/balance_nuevo
import sys, json, math, random, pathlib
import pandas as pd
sys.path.insert(0, '/content/grulla/experimentos/balance_nuevo')
from etapa19_buildspace_planner import all_spaces
from etapa19_skill_buildspace import enumerate_skill_builds, TECHNIQUE_TREES
from equipment_arc1_catalog import load_catalog


## 1. Dimensión exacta del espacio
Esto cuenta **todas** las combinaciones estructurales antes de simular.


In [ ]:
space_df = pd.DataFrame(all_spaces())
display(space_df[['stage','equipment_combinations_raw','skill_builds_per_root','skill_builds_all_roots','player_builds_raw','native_T0_T1_scenarios_raw']])


## 2. Monstruos traducidos al contrato nuevo
La traducción es **LAB**. HP/daño/técnica vienen del snapshot; Precision/Evasion preservan la dificultad de impacto legacy de referencia; DEF plana se asigna por rol/anatomía.


In [ ]:
MON = json.load(open('monster_arc1_new_contract_lab.json', encoding='utf-8'))
rows=[]
for mid,p in MON['profiles'].items():
    n=p['new_contract_lab']
    rows.append({'monster_id':mid,'name':p['name'],'stage':p['native_stage'],'role':p['role'],'hp':n['hp'],'precision':n['precision'],'evasion':n['evasion'],'defense':n['defense'],'tenacity':n['tenacity'],'damage':n['damage'],'survival':p['ai']['survival_t1']['kind']})
monster_df=pd.DataFrame(rows).sort_values(['stage','role','monster_id'])
display(monster_df)


## 3. Builds legales de habilidades
Se enumeran exactamente. Los puntos pueden guardarse; un Tramo superior requiere cualquier elección anterior en la misma técnica.


In [ ]:
skill_counts=[]
for stage in ['LianQi_I','LianQi_II','LianQi_III','LianQi_IV']:
    for root in TECHNIQUE_TREES:
        n=sum(1 for _ in enumerate_skill_builds(root,stage))
        skill_counts.append({'stage':stage,'root':root,'builds':n})
display(pd.DataFrame(skill_counts))


## 4. Estrategia de ejecución

No se intenta materializar 300 billones de filas. El pipeline usa:
1. conteo exhaustivo;
2. firmas mecánicas;
3. cobertura estratificada de todos los items/nodos;
4. Monte Carlo low-N;
5. refinamiento high-N de fronteras/anomalías.

La auditoría posterior de adquisición activará `ACQUISITION_LEGAL_ONLY` sin cambiar el motor.


In [ ]:
CONFIG = {
  'seed': 20260929,
  'stages': ['LianQi_I','LianQi_II','LianQi_III','LianQi_IV'],
  'adaptive_tiers_native': ['T0','T1'],
  'gear_mode': 'UNCONSTRAINED_STAGE_AVAILABLE_GEAR',
  'checkpoint_dir': '/content/etapa19_checkpoints',
  'low_n': 200,
  'high_n': 5000,
}
pathlib.Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
CONFIG


## 5. Piloto de Vestidura
Resultado guardado en el repositorio: `ETAPA19A_PILOTO_ARMADURA_MONSTRUOS_NATIVOS_2026-09-29.md`.

Sirve sólo para escala inicial; no sustituye el motor completo.


In [ ]:
pilot_path='/content/grulla/docs/experimentos/ETAPA19A_PILOTO_ARMADURA_MONSTRUOS_NATIVOS_2026-09-29.md'
print(open(pilot_path,encoding='utf-8').read())


## 6. Checkpoint y salidas
Cada bloque del motor masivo escribirá Parquet/CSV por etapa, monstruo, tier y política. Los resultados finales deben conservar `commit`, `seed`, `iterations`, `provenance` y multiplicidad estructural.
